```markdown
# Fraud Detection Notebook: Structural Overview

This notebook follows a three-stage pipeline:
1. **Data Ingestion & Graph Construction**: Loading the PaySim dataset and building a directed graph using `igraph` from all transactions (`df_graph`).
2. **Feature Engineering**: Extracting account-level topological features (Degree, PageRank) and mapping them back to individual transactions.
3. **Predictive Modeling**: Filtering the data into `df_clean` (specific to high-risk transaction types), balancing the classes via SMOTE, and training a Random Forest classifier.
```

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE

In [ ]:
!pip install -q igraph
import igraph as ig

print(ig.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 24.1 MB/s eta 0:00:00
1.0.0


In [ ]:

!pip install opendatasets --upgrade --quiet
import opendatasets as od

od.download("https://www.kaggle.com/datasets/ealaxi/paysim1")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: CloverAyush
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1


100%|██████████| 178M/178M [00:01<00:00, 96.0MB/s]


In [ ]:

df = pd.read_csv("paysim1/PS_20174392719_1491204439457_log.csv")

print("Dataset Shape (Rows, Columns):", df.shape)

Dataset Shape (Rows, Columns): (6362620, 11)


In [ ]:
print("--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- CLASS DISTRIBUTION (isFraud) ---")
print(df['isFraud'].value_counts())

--- MISSING VALUES ---
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

--- CLASS DISTRIBUTION (isFraud) ---
isFraud
0    6354407
1       8213
Name: count, dtype: int64


In [ ]:
df['transaction_id'] = [
    f'TX{i:08d}' for i in range(1, len(df) + 1)
]

In [ ]:
df_graph = df.copy()

df_graph.drop(columns=['isFlaggedFraud'], inplace=True, errors='ignore')
df_graph.drop(columns=['isFraud'], inplace=True, errors='ignore')

df_graph.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,transaction_id
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,TX00000001
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,TX00000002
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,TX00000003
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,TX00000004
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,TX00000005


```markdown
### Global Graph Data Preparation
`df_graph` is used here to maintain the full network topology, ensuring that degrees and PageRank values are calculated based on the entire set of available relationships before filtering for specific transaction types.
```

In [ ]:
#  filtered table with only TRANSFER and CASH_OUT
df_clean = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

# remaining row count
print("Remaining rows:", len(df_clean))

Remaining rows: 2770409


```markdown
### Modeling Dataset Alignment
`df_clean` is derived from the original `df` to focus exclusively on 'TRANSFER' and 'CASH_OUT' transactions. Features calculated from the global graph (`df_graph`) are then mapped onto this subset to maintain the contextual importance of account behavior.
```

In [ ]:
df_clean.drop(columns=['isFlaggedFraud'], inplace=True, errors='ignore')
# Calculate the exact mathematical error in the sender's account


df_clean['errorBalanceOrig'] = df_clean['newbalanceOrig'] + df_clean['amount'] - df_clean['oldbalanceOrg']

# Calculate the exact mathematical error in the receiver's account
df_clean['errorBalanceDest'] = df_clean['oldbalanceDest'] + df_clean['amount'] - df_clean['newbalanceDest']
df_clean.shape

(2770409, 13)

In [ ]:
accounts = pd.unique(
    df_graph[['nameOrig', 'nameDest']].values.ravel()
)

account_to_id = pd.Series(
    range(len(accounts)),
    index=accounts
)

print("Accounts:", len(accounts))
print(account_to_id.head())

Accounts: 9073900
C1231006815    0
M1979787155    1
C1666544295    2
M2044282225    3
C1305486145    4
dtype: int64


In [ ]:
sample_orig = df_graph['nameOrig'].iloc[0]
sample_dest = df_graph['nameDest'].iloc[0]

print("Original:", sample_orig, "→", sample_dest)
print("Mapped:",
      account_to_id[sample_orig],
      "→",
      account_to_id[sample_dest])

Original: C1231006815 → M1979787155
Mapped: 0 → 1


In [ ]:
src = df_graph['nameOrig'].map(account_to_id).to_numpy()
dst = df_graph['nameDest'].map(account_to_id).to_numpy()

In [ ]:
G = ig.Graph(
    n=len(accounts),
    edges=np.column_stack((src, dst)),
    directed=True
)

In [ ]:
#============================pagerank===========================
pagerank = np.asarray(
    G.pagerank(
        directed=True,
        weights=None
    ),
    dtype=np.int32
)
"""
G.es["weight"] = df_graph["amount"].to_numpy()
print(G.es["weight"][:5])
print(df_graph["amount"].head().to_list())
"""

'\nG.es["weight"] = df_graph["amount"].to_numpy()\nprint(G.es["weight"][:5])\nprint(df_graph["amount"].head().to_list())\n'

In [ ]:

# --------------------------------------------------
# STATIC ACCOUNT-LEVEL GRAPH FEATURES
# --------------------------------------------------

# Number of vertices in the graph
num_vertices = G.vcount()

# Account IDs in the exact same order as igraph vertices
accounts = np.array(accounts)

assert len(accounts) == num_vertices

# Degree features
in_degree = np.asarray(G.indegree(), dtype=np.int32)
out_degree = np.asarray(G.outdegree(), dtype=np.int32)

# Flow ratio
# Incoming relationships relative to outgoing relationships
flow_ratio = in_degree.astype(np.float32) / np.maximum(out_degree, 1)

# Unweighted PageRank
pagerank = np.asarray(
    G.pagerank(
        directed=True,
        weights=None
    ),
    dtype=np.float32
)

# --------------------------------------------------
# BUILD CANONICAL ACCOUNT FEATURE TABLE
# --------------------------------------------------

account_graph_features = pd.DataFrame({
    'account_id': accounts,
    'vertex_id': np.arange(num_vertices, dtype=np.int32),
    'in_degree': in_degree,
    'out_degree': out_degree,
    'flow_ratio': flow_ratio,
    'pagerank': pagerank
})

print("Account graph feature table:")
print(account_graph_features.head())

print("\nShape:", account_graph_features.shape)

print("\nMissing values:")
print(account_graph_features.isna().sum())

Account graph feature table:
    account_id  vertex_id  in_degree  out_degree  flow_ratio      pagerank
0  C1231006815          0          0           1         0.0  6.904254e-08
1  M1979787155          1          1           0         1.0  1.277287e-07
2  C1666544295          2          0           1         0.0  6.904254e-08
3  M2044282225          3          1           0         1.0  1.277287e-07
4  C1305486145          4          0           1         0.0  6.904254e-08

Shape: (9073900, 6)

Missing values:
account_id    0
vertex_id     0
in_degree     0
out_degree    0
flow_ratio    0
pagerank      0
dtype: int64


In [ ]:
# --------------------------------------------------
# ATTACH STATIC GRAPH FEATURES TO TRANSACTIONS
# --------------------------------------------------

df_graph['orig_id'] = src
df_graph['dest_id'] = dst

# Sender / origin features
df_graph['orig_in_degree'] = in_degree[src]
df_graph['orig_out_degree'] = out_degree[src]
df_graph['orig_flow_ratio'] = flow_ratio[src]
df_graph['orig_pagerank'] = pagerank[src]

# Receiver / destination features
df_graph['dest_in_degree'] = in_degree[dst]
df_graph['dest_out_degree'] = out_degree[dst]
df_graph['dest_flow_ratio'] = flow_ratio[dst]
df_graph['dest_pagerank'] = pagerank[dst]

print(df_graph[
    [
        'nameOrig',
        'nameDest',
        'orig_id',
        'dest_id',
        'orig_in_degree',
        'orig_out_degree',
        'orig_flow_ratio',
        'orig_pagerank',
        'dest_in_degree',
        'dest_out_degree',
        'dest_flow_ratio',
        'dest_pagerank'
    ]
].head())

df_graph.head()

      nameOrig     nameDest  orig_id  dest_id  orig_in_degree  \
0  C1231006815  M1979787155        0        1               0   
1  C1666544295  M2044282225        2        3               0   
2  C1305486145   C553264065        4        5               0   
3   C840083671    C38997010        6        7               0   
4  C2048537720  M1230701703        8        9               0   

   orig_out_degree  orig_flow_ratio  orig_pagerank  dest_in_degree  \
0                1              0.0              0               1   
1                1              0.0              0               1   
2                1              0.0              0              44   
3                1              0.0              0              41   
4                1              0.0              0               1   

   dest_out_degree  dest_flow_ratio  dest_pagerank  
0                0              1.0              0  
1                0              1.0              0  
2                0           

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,transaction_id,orig_id,dest_id,orig_in_degree,orig_out_degree,orig_flow_ratio,orig_pagerank,dest_in_degree,dest_out_degree,dest_flow_ratio,dest_pagerank
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,TX00000001,0,1,0,1,0.0,0,1,0,1.0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,TX00000002,2,3,0,1,0.0,0,1,0,1.0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,TX00000003,4,5,0,1,0.0,0,44,0,44.0,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,TX00000004,6,7,0,1,0.0,0,41,0,41.0,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,TX00000005,8,9,0,1,0.0,0,1,0,1.0,0


In [ ]:
# --------------------------------------------------
# INSPECT CURRENT TRANSACTION COLUMNS
# --------------------------------------------------

print(df_graph.columns.tolist())

# Check whether transaction_id is unique
print("Total transactions:", len(df_graph))
print("Unique transaction IDs:", df_graph['transaction_id'].nunique())

print(
    "Duplicate transaction IDs:",
    df_graph['transaction_id'].duplicated().sum()
)

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'transaction_id', 'orig_id', 'dest_id', 'orig_in_degree', 'orig_out_degree', 'orig_flow_ratio', 'orig_pagerank', 'dest_in_degree', 'dest_out_degree', 'dest_flow_ratio', 'dest_pagerank']
Total transactions: 6362620
Unique transaction IDs: 6362620
Duplicate transaction IDs: 0


In [ ]:
print("Vertices:", G.vcount())
print("Edges:", G.ecount())
print("Directed:", G.is_directed())

Vertices: 9073900
Edges: 6362620
Directed: True


In [ ]:
###============================EDA===================================

in_degree = np.array(G.indegree())
out_degree = np.array(G.outdegree())

print("In-degree:")
print(pd.Series(in_degree).describe())

print("\nOut-degree:")
print(pd.Series(out_degree).describe())

In-degree:
count    9.073900e+06
mean     7.012001e-01
std      2.712253e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      1.130000e+02
dtype: float64

Out-degree:
count    9.073900e+06
mean     7.012001e-01
std      4.599723e-01
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      3.000000e+00
dtype: float64


In [ ]:
###--------------------------EDA-----------------------------------

print("Accounts with in-degree > 1:",
      np.sum(in_degree > 1))

print("Accounts with out-degree > 1:",
      np.sum(out_degree > 1))

print("Accounts with both > 1:",
      np.sum((in_degree > 1) & (out_degree > 1)))

Accounts with in-degree > 1: 459658
Accounts with out-degree > 1: 9298
Accounts with both > 1: 2


In [ ]:

###------------------------EDA-------------------------
n = len(in_degree)

print(
    "In-degree > 1:",
    np.sum(in_degree > 1) / n * 100,
    "%"
)

print(
    "Out-degree > 1:",
    np.sum(out_degree > 1) / n * 100,
    "%"
)

In-degree > 1: 5.065715954550965 %
Out-degree > 1: 0.10246972084770607 %


In [ ]:
###============================EDA=================================

df_degree_test = df_clean[['transaction_id', 'nameOrig', 'nameDest', 'isFraud']].copy()

df_degree_test['orig_id'] = df_degree_test['nameOrig'].map(account_to_id)
df_degree_test['dest_id'] = df_degree_test['nameDest'].map(account_to_id)

df_degree_test['orig_in_degree'] = in_degree[df_degree_test['orig_id'].to_numpy()]
df_degree_test['orig_out_degree'] = out_degree[df_degree_test['orig_id'].to_numpy()]

df_degree_test['dest_in_degree'] = in_degree[df_degree_test['dest_id'].to_numpy()]
df_degree_test['dest_out_degree'] = out_degree[df_degree_test['dest_id'].to_numpy()]

print(
    df_degree_test.groupby('isFraud')[
        ['orig_in_degree',
         'orig_out_degree',
         'dest_in_degree',
         'dest_out_degree']
    ].mean()
)

print(
    df_degree_test.groupby('isFraud')[
        ['orig_in_degree',
         'orig_out_degree',
         'dest_in_degree',
         'dest_out_degree']
    ].median()
)

print(
    df_degree_test.groupby('isFraud')['dest_in_degree']
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
)


         orig_in_degree  orig_out_degree  dest_in_degree  dest_out_degree
isFraud                                                                  
0              0.002129         1.002935       16.763414         0.003166
1              0.000000         1.003409        8.095337         0.002192
         orig_in_degree  orig_out_degree  dest_in_degree  dest_out_degree
isFraud                                                                  
0                   0.0              1.0            14.0              0.0
1                   0.0              1.0             4.0              0.0
             count       mean        std  min  25%   50%   75%   90%   95%  \
isFraud                                                                      
0        2762196.0  16.763414  12.355521  1.0  7.0  14.0  23.0  34.0  41.0   
1           8213.0   8.095337   9.883579  1.0  1.0   4.0  12.0  22.0  28.0   

           99%    max  
isFraud                
0        54.00  113.0  
1        44.88   89.0  

In [ ]:
#####=============================Weighted check============================
"""
weighted_pagerank = G.pagerank(
    directed=True,
    weights="weight"
)
"""

'\nweighted_pagerank = G.pagerank(\n    directed=True,\n    weights="weight"\n)\n'

In [ ]:

"""
print(
    pd.Series(weighted_pagerank).describe()
)

df_clean['dest_weighted_pagerank'] = [
    weighted_pagerank[int(i)]
    for i in df_clean['dest_id']
]

weighted_comparison = (
    df_clean
    .groupby('isFraud')['dest_weighted_pagerank']
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print(weighted_comparison)

weighted_means = df_clean.groupby(
    'isFraud'
)['dest_weighted_pagerank'].mean()

print("\nWeighted PageRank means:")
print(weighted_means)

print(
    "\nFraud / Legitimate:",
    weighted_means[1] / weighted_means[0]
)
"""

'\nprint(\n    pd.Series(weighted_pagerank).describe()\n)\n\ndf_clean[\'dest_weighted_pagerank\'] = [\n    weighted_pagerank[int(i)]\n    for i in df_clean[\'dest_id\']\n]\n\nweighted_comparison = (\n    df_clean\n    .groupby(\'isFraud\')[\'dest_weighted_pagerank\']\n    .describe(\n        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]\n    )\n)\n\nprint(weighted_comparison)\n\nweighted_means = df_clean.groupby(\n    \'isFraud\'\n)[\'dest_weighted_pagerank\'].mean()\n\nprint("\nWeighted PageRank means:")\nprint(weighted_means)\n\nprint(\n    "\nFraud / Legitimate:",\n    weighted_means[1] / weighted_means[0]\n)\n'

In [ ]:
###============================Adding dest_id, pagerank and indegree to df_clean============================

df_clean['dest_id'] = df_clean['nameDest'].map(account_to_id)

df_clean['dest_in_degree'] = in_degree[
    df_clean['dest_id'].to_numpy()
]

df_clean['dest_pagerank'] = [
    pagerank[int(i)]
    for i in df_clean['dest_id']
]

print(df_clean[['nameDest', 'dest_id', 'dest_pagerank']].head())

print("Missing IDs:", df_clean['dest_id'].isna().sum())
print("Missing PageRank:", df_clean['dest_pagerank'].isna().sum())

df_clean.shape

       nameDest  dest_id  dest_pagerank
2    C553264065        5              0
3     C38997010        7              0
15   C476402209       31              0
19  C1100439041       39              0
24   C932583850       49              0
Missing IDs: 0
Missing PageRank: 0


(2770409, 16)

In [ ]:
print("Graph edges:", G.ecount())
print("Amounts:", len(df_graph['amount']))

Graph edges: 6362620
Amounts: 6362620


In [ ]:
###===============================EDA=================================
# PageRank comparison: Legitimate vs Fraud

pagerank_comparison = (
    df_clean
    .groupby('isFraud')['dest_pagerank']
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

print(pagerank_comparison)

print("\nMean PageRank:")
print(df_clean.groupby('isFraud')['dest_pagerank'].mean())

print("\nMedian PageRank:")
print(df_clean.groupby('isFraud')['dest_pagerank'].median())

print("\nMaximum PageRank:")
print(df_clean.groupby('isFraud')['dest_pagerank'].max())

print("\nFraud vs Legitimate PageRank ratio:")

legit_mean = df_clean.loc[
    df_clean['isFraud'] == 0, 'dest_pagerank'
].mean()

fraud_mean = df_clean.loc[
    df_clean['isFraud'] == 1, 'dest_pagerank'
].mean()

print("Legitimate mean:", legit_mean)
print("Fraud mean:", fraud_mean)
print("Fraud / Legitimate:", fraud_mean / legit_mean)

             count  mean  std  min  25%  50%  75%  90%  95%  99%  max
isFraud                                                              
0        2762196.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
1           8213.0   0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

Mean PageRank:
isFraud
0    0.0
1    0.0
Name: dest_pagerank, dtype: float64

Median PageRank:
isFraud
0    0.0
1    0.0
Name: dest_pagerank, dtype: float64

Maximum PageRank:
isFraud
0    0
1    0
Name: dest_pagerank, dtype: int32

Fraud vs Legitimate PageRank ratio:
Legitimate mean: 0.0
Fraud mean: 0.0
Fraud / Legitimate: nan


/tmp/ipykernel_1220/3077446982.py:33: RuntimeWarning: invalid value encountered in scalar divide
  print("Fraud / Legitimate:", fraud_mean / legit_mean)


In [ ]:

print(df_clean['dest_in_degree'].describe())

count    2.770409e+06
mean     1.673772e+01
std      1.235791e+01
min      1.000000e+00
25%      7.000000e+00
50%      1.400000e+01
75%      2.300000e+01
max      1.130000e+02
Name: dest_in_degree, dtype: float64


In [ ]:
#=======================================EDA===============================
pr = np.array(pagerank)

print("PageRank length:", len(pr))
print(pd.Series(pr).describe())
print("Min:", pr.min())
print("Max:", pr.max())
print("Non-zero:", np.count_nonzero(pr))

PageRank length: 9073900
count    9073900.0
mean           0.0
std            0.0
min            0.0
25%            0.0
50%            0.0
75%            0.0
max            0.0
dtype: float64
Min: 0
Max: 0
Non-zero: 0


In [ ]:
import pickle

# Account-level graph features
account_graph_features.to_parquet(
    "account_graph_features.parquet",
    index=False
)

# Transaction-level graph data
df_graph.to_parquet(
    "df_graph.parquet",
    index=False
)

# Account → vertex ID mapping
with open("account_to_id.pkl", "wb") as f:
    pickle.dump(account_to_id, f)

# Static igraph graph
G.write_pickle("transaction_graph.pkl")

print("Graph artifacts saved.")

Graph artifacts saved.


In [ ]:
import os

size_mb = os.path.getsize("df_graph.parquet") / (1024**2)

print(f"df_graph.parquet: {size_mb:.2f} MB")

df_graph.parquet: 357.05 MB


In [ ]:
# CELL 7: Translate text categories into numbers
encoder = LabelEncoder()
df_clean['type'] = encoder.fit_transform(df_clean['type'])

# Print out the data types of every column to prove zero text remains
print("--- COLUMN DATA TYPES ---")
print(df_clean.dtypes)
df_clean.head()

--- COLUMN DATA TYPES ---
step                  int64
type                  int64
amount              float64
nameOrig             object
oldbalanceOrg       float64
newbalanceOrig      float64
nameDest             object
oldbalanceDest      float64
newbalanceDest      float64
isFraud               int64
transaction_id       object
errorBalanceOrig    float64
errorBalanceDest    float64
dest_id               int64
dest_in_degree        int64
dest_pagerank         int32
dtype: object


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,transaction_id,errorBalanceOrig,errorBalanceDest,dest_id,dest_in_degree,dest_pagerank
2,1,1,181.00,C1305486145,181.0,0.0,C553264065,0.0,0.00,1,TX00000003,0.00,181.0,5,44,0
3,1,0,181.00,C840083671,181.0,0.0,C38997010,21182.0,0.00,1,TX00000004,0.00,21363.0,7,41,0
15,1,0,229133.94,C905080434,15325.0,0.0,C476402209,5083.0,51513.44,0,TX00000016,213808.94,182703.5,31,41,0
19,1,1,215310.30,C1670993182,705.0,0.0,C1100439041,22425.0,0.00,0,TX00000020,214605.30,237735.3,39,47,0
24,1,1,311685.89,C1984094095,10835.0,0.0,C932583850,6267.0,2719172.89,0,TX00000025,300850.89,-2401220.0,49,82,0


In [ ]:
X = df_clean.drop(
    columns=[
        'isFraud',
        'transaction_id',
        'nameOrig',
        'nameDest',
        'dest_id',
        'dest_in_degree',


    ]
)

X.head()

,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,errorBalanceOrig,errorBalanceDest,dest_pagerank
2,1,1,181.00,181.0,0.0,0.0,0.00,0.00,181.0,0
3,1,0,181.00,181.0,0.0,21182.0,0.00,0.00,21363.0,0
15,1,0,229133.94,15325.0,0.0,5083.0,51513.44,213808.94,182703.5,0
19,1,1,215310.30,705.0,0.0,22425.0,0.00,214605.30,237735.3,0
24,1,1,311685.89,10835.0,0.0,6267.0,2719172.89,300850.89,-2401220.0,0


In [ ]:
# Define features (X) and target (y)
X = df_clean.drop(
    columns=[
        'isFraud',
        'transaction_id',
        'nameOrig',
        'nameDest',
        'dest_id',
        'dest_in_degree'
    ]
)

y = df_clean['isFraud']
y = df_clean['isFraud']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# CELL 9: Balance the Training Set using SMOTE

print("--- BEFORE SMOTE (Training Set) ---")
print(y_train.value_counts())

# 1. Initialize the SMOTE tool
smote = SMOTE( sampling_strategy=0.1, random_state=42)

# 2. Generate synthetic fraud examples ONLY in the training set

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print("\n--- AFTER SMOTE (Balanced Training Set) ---")
print(y_train_balanced.value_counts())

--- BEFORE SMOTE (Training Set) ---
isFraud
0    1933537
1       5749
Name: count, dtype: int64

--- AFTER SMOTE (Balanced Training Set) ---
isFraud
0    1933537
1     193353
Name: count, dtype: int64


In [ ]:
print(X.shape)
print(X.dtypes)
print(X.isna().sum().sum())

print("dest_in_degree" in X.columns)
print("dest_id" in X.columns)
print("nameOrig" in X.columns)
print("nameDest" in X.columns)

(2770409, 10)
step                  int64
type                  int64
amount              float64
oldbalanceOrg       float64
newbalanceOrig      float64
oldbalanceDest      float64
newbalanceDest      float64
errorBalanceOrig    float64
errorBalanceDest    float64
dest_pagerank         int32
dtype: object
0
False
False
False
False


In [53]:
# CELL 10: Train the Random Forest Algorithm on the Balanced Data
from sklearn.ensemble import RandomForestClassifier
import time
import joblib # Import joblib for saving and loading models


# 1. Set up the Random Forest with our optimized rules

model = RandomForestClassifier(
    n_estimators=200,      # 200 trees in the forest
    max_depth=10,         # Prevent overfitting
    n_jobs=-1,            # Use all Colab CPU cores for speed!
    random_state=42       # Ensure consistent results
)

print("Training started on 3.86 million rows! ")
start_time = time.time()

# 2. Train the model (The AI is learning right now!)
model.fit(X_train_balanced, y_train_balanced)

end_time = time.time()
print(f"\n--- TRAINING COMPLETE! ---")
print(f"Time taken: {round(end_time - start_time, 2)} seconds")
print("Your Random Forest model is officially trained and ready to hunt fraudsters!")

# Export Production Model
print("\nExporting production model...")

joblib.dump(model, "fraud_model.joblib")
joblib.dump(encoder, "label_encoder.joblib")
joblib.dump(X.columns.tolist(), "feature_columns.joblib")

print("✅ fraud_model_graph.joblib")
print("✅ label_encoder_graph.joblib")
print("✅ feature_columns_graph.joblib")
print("\nProduction model exported successfully!")


Training started on 3.86 million rows! 


KeyboardInterrupt: 

In [ ]:
print(X.columns.tolist())

print("dest_in_degree in X:",
      'dest_in_degree' in X.columns)

print("Model features:",
      model.n_features_in_)

print(
    pd.Series(
        model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)
)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

def evaluate_model(model, X_eval, y_eval):
    # Predictions
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]

    print("=" * 60)
    print("MODEL EVALUATION")
    print("=" * 60)

    print(f"Evaluating rows: {len(y_eval)}\n")

    print(f"Precision : {precision_score(y_eval, y_pred):.6f}")
    print(f"Recall    : {recall_score(y_eval, y_pred):.6f}")
    print(f"F1 Score  : {f1_score(y_eval, y_pred):.6f}")
    print(f"ROC-AUC   : {roc_auc_score(y_eval, y_prob):.6f}")
    print(f"PR-AUC    : {average_precision_score(y_eval, y_prob):.6f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_eval, y_pred))

    print("\nClassification Report:")
    print(classification_report(
        y_eval,
        y_pred,
        target_names=["Legitimate", "Fraud"]
    ))

    # FP / FN count
    cm = confusion_matrix(y_eval, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print("\nFalse Positives :", fp)
    print("False Negatives :", fn)

    return {
        "precision": precision_score(y_eval, y_pred),
        "recall": recall_score(y_eval, y_pred),
        "f1": f1_score(y_eval, y_pred),
        "roc_auc": roc_auc_score(y_eval, y_prob),
        "pr_auc": average_precision_score(y_eval, y_prob),
        "false_positives": fp,
        "false_negatives": fn
    }

results = evaluate_model(
    model,
    X_test,
    y_test
)

print(results)